In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel


In [2]:
BATCH_SIZE = 16
LEARNING_RATE = 2e-5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4
PATIENCE = 3

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}")

Using device: cuda
Batch size: 16, Learning rate: 2e-05


In [3]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1
...,...,...
8525,any enjoyment will be hinge from a personal th...,0
8526,if legendary shlockmeister ed wood had ever ma...,0
8527,hardly a nuanced portrait of a young woman's b...,0
8528,"interminably bleak , to say nothing of boring .",0


In [4]:
class BinaryClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),

            'labels': torch.tensor([label], dtype=torch.float)
        }


In [5]:
class BertForBinaryClassification(nn.Module):
    def __init__(self):
        super(BertForBinaryClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(768, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        output = self.classifier(pooled_output)
        return torch.sigmoid(output)
    

In [6]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [7]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            train_loss += loss.item()
            preds = (outputs > 0.5).float().cpu().numpy()
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds).flatten()
        train_true = np.array(train_true).flatten()
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (outputs > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds).flatten()
        val_true = np.array(val_true).flatten()
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [8]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []  # To track time per sample

    start_test = perf_counter()

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            # Process one sample at a time (for each input in the batch)
            for i in range(input_ids.size(0)):  # Process each sample in the batch
                # Get individual sample
                input_id = input_ids[i].unsqueeze(0)  # Add batch dimension
                attention_mask_sample = attention_mask[i].unsqueeze(0)  # Add batch dimension
                label = labels[i].item()

                # Track the time per sample
                start_time = perf_counter()
                
                # Make prediction
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)
                pred = (output > 0.5).float().cpu().numpy().flatten()[0]
                
                # Append results
                predictions.append(pred)
                true_labels.append(label)

                # Track classification time for each sample
                classification_times.append(perf_counter() - start_time)

    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)
    true_labels = np.array(true_labels)

    # Now you can calculate metrics
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)

    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)

    return predictions, true_labels


In [ ]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

train_dataset = BinaryClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = BinaryClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = BinaryClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
results = []

# Grid search loop
for seed in seeds:
    torch.manual_seed(seed)
    model = BertForBinaryClassification().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()
    
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
    save_path = f'results/bert_binary2_bs{BATCH_SIZE}_lr{LEARNING_RATE}_seed{seed}.pt'

    # Train
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
         {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)
    
    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval
    
    # Load best model
    model.load_state_dict(torch.load(save_path))
    
    # Evaluate
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, test_retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = test_retval
    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)
    
    # Store individual results for this seed
    results.append({
        'seed': seed,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_acc': train_acc,
        'train_precisions': train_precisions.tolist(),
        'train_recalls': train_recalls.tolist(),
        'train_f1s': train_f1s.tolist(),
        'max_memory_usage_train': max_memory_usage_train,
        'max_vram_usage_train': max_vram_usage_train,
        'total_train_time': total_train_time,
        'val_acc': val_acc,
        'val_precisions': val_precisions.tolist(),
        'val_recalls': val_recalls.tolist(),
        'val_f1s': val_f1s.tolist(),
        'test_acc': test_acc,
        'test_precisions': test_precisions.tolist(),
        'test_recalls': test_recalls.tolist(),
        'test_f1s': test_f1s.tolist(),
        'max_memory_usage_test': max_memory_usage_test,
        'max_vram_usage_test': max_vram_usage_test,
        'total_test_time': total_time_test
    })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/534 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Epoch 1/4 - Train Loss: 0.3919, Acc: 0.8251, F1: [0.82663258 0.8235155 ]
Epoch 1/4 - Val Loss: 0.3304, Acc: 0.8583, F1: [0.85494717 0.86159487]
Model saved!


Epoch 2/4 - Train Loss: 0.1907, Acc: 0.9307, F1: [0.93160514 0.92980164]
Epoch 2/4 - Val Loss: 0.3832, Acc: 0.8377, F1: [0.82542886 0.84837862]


Epoch 3/4 - Train Loss: 0.1007, Acc: 0.9666, F1: [0.96666277 0.96651392]
Epoch 3/4 - Val Loss: 0.4173, Acc: 0.8593, F1: [0.85795455 0.8605948 ]


Epoch 4/4 - Train Loss: 0.0539, Acc: 0.9841, F1: [0.98410101 0.98401129]
Epoch 4/4 - Val Loss: 0.5423, Acc: 0.8565, F1: [0.85302594 0.85976169]
Early stopping triggered


...

Epoch 2/4 - Train Loss: 0.1933, Acc: 0.9293, F1: [0.93002205 0.92857989]
Epoch 2/4 - Val Loss: 0.3588, Acc: 0.8677, F1: [0.87004608 0.86532951]


Epoch 3/4 - Train Loss: 0.0882, Acc: 0.9715, F1: [0.97158227 0.971442  ]
Epoch 3/4 - Val Loss: 0.4268, Acc: 0.8724, F1: [0.87636364 0.86821705]


Epoch 4/4 - Train Loss: 0.0521, Acc: 0.9851, F1: [0.98511661 0.98510613]
Epoch 4/4 - Val Loss: 0.5324, Acc: 0.8621, F1: [0.85545723 0.86816143]
Early stopping triggered


C:\Users\Rafael\AppData\Local\Temp\ipykernel_23376\596530835.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafae

Test Time: 10.09 seconds
Test Metrics:
Accuracy: 0.8583489681050657
F1s: [0.86334842 0.85296981]
Precisions: [0.83391608 0.88663968]
Recalls: [0.89493433 0.8217636 ]


In [ ]:
df = pd.DataFrame(results)
df.to_csv('results/bert_binary2.csv', index=False)

In [11]:
df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,16,0.00002,0.825088,"[0.8193964524303156, 0.8309859154929577]","[0.8339976553341149, 0.8161781946072685]","[0.826632581919591, 0.8235154956233736]",1335.171875,2528.859863,382.974231,...,"[0.8759842519685039, 0.8422939068100358]","[0.8348968105065666, 0.8818011257035647]","[0.8549471661863592, 0.8615948670944088]",0.845216,"[0.8407407407407408, 0.8498098859315589]","[0.851782363977486, 0.8386491557223265]","[0.8462255358807083, 0.8441926345609065]",1279.988281,1705.130859,9.976735
1,3,16,0.00002,0.836108,"[0.8210526315789474, 0.8526445264452645]","[0.8595545134818289, 0.8126611957796014]","[0.8398625429553265, 0.8321728691476591]",1346.320312,2534.109863,441.400862,...,"[0.8645038167938931, 0.8523985239852399]","[0.849906191369606, 0.8667917448405253]","[0.8571428571428571, 0.8595348837209302]",0.853659,"[0.840867992766727, 0.8674463937621832]","[0.8724202626641651, 0.8348968105065666]","[0.856353591160221, 0.8508604206500956]",1346.296875,1704.130859,9.566090
2,5,16,0.00002,0.832825,"[0.832046783625731, 0.8336075205640423]","[0.8339976553341149, 0.8316529894490036]","[0.8330210772833724, 0.8326291079812207]",1346.597656,2529.859863,483.820780,...,"[0.8597785977859779, 0.8721374045801527]","[0.874296435272045, 0.8574108818011257]","[0.8669767441860465, 0.8647114474929044]",0.858349,"[0.833916083916084, 0.8866396761133604]","[0.8949343339587242, 0.8217636022514071]","[0.8633484162895928, 0.8529698149951315]",1346.312500,1704.755859,10.631150
